# UAV-IRE Training — WeedyRice-RGBMS-DB
**Tesis: UAV-IRE Super-Resolution untuk Segmentasi Gulma Lahan Persawahan**  
Muhamad Syaiful Huda | NRP 6002241012 | ITS Surabaya 2025

| `EXPERIMENT` | Skenario | Deskripsi |
|---|---|---|
| `'baseline'` | SR1 | IRE baseline (tanpa modul UAV) |
| `'no_nrdb'`  | SR2 | UAV-IRE tanpa NRDB |
| `'no_mbcm'`  | SR3 | UAV-IRE tanpa MBCM |
| `'no_ega'`   | SR4 | UAV-IRE tanpa EGA |
| `'no_vsd'`   | SR5 | UAV-IRE tanpa VSD |
| `'full'`     | SR6 | **UAV-IRE Lengkap** |

## ⚙️ Konfigurasi — UBAH DI SINI

In [ ]:
EXPERIMENT      = 'full'   # ← GANTI untuk ablation lain
TOTAL_EPOCHS    = 50
BATCH_SIZE      = 4        # T4: 4, P100: 8
GT_PATCH_SIZE   = 256      # patch training dari gambar 5280x3956
EVAL_PATCH_SIZE = 512      # patch center-crop untuk validasi
NUM_RRDB        = 23
LR_G            = 1e-4
LR_DECAY_EPOCH  = 25
LAMBDA_REC, LAMBDA_PERC, LAMBDA_ADV = 1.0, 1.0, 0.1
LAMBDA_EDGE, LAMBDA_VSD              = 0.05, 0.1

GITHUB_REPO = 'https://github.com/SyaifulHuda25/uav-ire-final.git'  # GANTI
OUTPUT_DIR  = '/kaggle/working/experiments'

# Path WeedyRice-RGBMS-DB (sesuai Kaggle Data Explorer Anda)
_BASE = (
    '/kaggle/input/datasets/muhamadsyaifulhuda/'
    'weedy-rice-uav-segmentation/'
    'A Dataset of Aligned RGB and Multispectral UAV Ima/'
    'WeedyRice-RGBMS-DB'
)
DATASET_ROOT = f'{_BASE}/WeedyRice-RGBMS-DB'  # berisi RGB/, Masks/, dll
TRAIN_LIST   = f'{_BASE}/train_list.txt'
VAL_LIST     = f'{_BASE}/val_list.txt'
TEST_LIST    = f'{_BASE}/test_list.txt'

# Pretrained Real-ESRGAN — diisi otomatis di sel berikutnya
PRETRAINED_PATH = ''

print(f'Experiment: {EXPERIMENT} | Epochs: {TOTAL_EPOCHS} | Batch: {BATCH_SIZE}')
print(f'Dataset root: {DATASET_ROOT}')

## 1. Setup

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'pandas', '-q'], check=True)

REPO_DIR = '/kaggle/working/uav-ire'
if not os.path.isdir(REPO_DIR):
    assert os.system(f'git clone {GITHUB_REPO} {REPO_DIR}') == 0, 'Clone gagal — cek GITHUB_REPO'
else:
    os.system(f'cd {REPO_DIR} && git pull')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

import glob
print(f'RGB files  : {len(glob.glob(f"{DATASET_ROOT}/RGB/*.JPG"))}')
print(f'Mask files : {len(glob.glob(f"{DATASET_ROOT}/Masks/*.png"))}')
print(f'train_list : {os.path.isfile(TRAIN_LIST)}')

In [ ]:
# Download pretrained Real-ESRGAN jika belum ada
PRETRAINED_LOCAL = '/kaggle/working/RealESRGAN_x4plus.pth'
if not os.path.isfile(PRETRAINED_LOCAL):
    print('Downloading Real-ESRGAN pretrained...')
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    os.system(f'wget -q "{url}" -O {PRETRAINED_LOCAL}')
print(f'Pretrained: {PRETRAINED_LOCAL} ({os.path.getsize(PRETRAINED_LOCAL)/1e6:.0f} MB)')
PRETRAINED_PATH = PRETRAINED_LOCAL

In [ ]:
# Estimasi waktu
with open(TRAIN_LIST) as f:
    n_train = sum(1 for l in f if l.strip())
with open(VAL_LIST) as f:
    n_val = sum(1 for l in f if l.strip())

est_iter   = n_train // BATCH_SIZE
total_iter = TOTAL_EPOCHS * est_iter
est_hours  = total_iter / 2.5 / 3600
print(f'Train:{n_train} | Val:{n_val} | ~{est_iter} iter/epoch | Total:{total_iter:,} | ~{est_hours:.1f}h')

## 2. Trainer (Auto-Resume)

In [ ]:
from training.epoch_trainer import EpochTrainer

trainer = EpochTrainer(
    experiment      = EXPERIMENT,
    dataset_root    = DATASET_ROOT,
    train_list      = TRAIN_LIST,
    val_list        = VAL_LIST,
    save_dir        = OUTPUT_DIR,
    total_epochs    = TOTAL_EPOCHS,
    batch_size      = BATCH_SIZE,
    gt_patch_size   = GT_PATCH_SIZE,
    eval_patch_size = EVAL_PATCH_SIZE,
    num_rrdb        = NUM_RRDB,
    lr_g            = LR_G,
    lr_decay_epoch  = LR_DECAY_EPOCH,
    use_mask        = True,
    lambda_rec=LAMBDA_REC, lambda_perc=LAMBDA_PERC,
    lambda_adv=LAMBDA_ADV, lambda_edge=LAMBDA_EDGE, lambda_vsd=LAMBDA_VSD,
    pretrained_path = PRETRAINED_PATH,
    log_iter_freq   = 20,
    val_epoch_freq  = 2,
    save_epoch_freq = 1,   # tiap epoch untuk resume
    vis_epoch_freq  = 5,
    use_amp=True, num_workers=2,
)

print(f'Start epoch: {trainer.start_epoch}/{TOTAL_EPOCHS}')
if trainer.start_epoch > 1:
    print(f'AUTO-RESUME dari epoch {trainer.start_epoch} '
          f'(sisa {TOTAL_EPOCHS - trainer.start_epoch + 1} epoch)')

## 3. Training

In [ ]:
history = trainer.train()
print('Training selesai!')

## 4. Grafik Train Loss vs Valid Loss

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

exp_dir  = trainer.save_dir
exp_name = trainer.exp_name

with open(os.path.join(exp_dir, 'history.json')) as f:
    data = json.load(f)

epochs    = data['epochs']
tr_loss   = data['train'].get('total', [])
val_loss  = data['val'].get('val_loss', [])
psnr_vals = data['val'].get('psnr', [])
ssim_vals = data['val'].get('ssim', [])
val_ep    = [e for e in epochs if e % trainer.val_epoch_freq == 0][:len(val_loss)]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'UAV-IRE — {exp_name}\nWeedyRice-RGBMS-DB | {TOTAL_EPOCHS} Epoch',
             fontsize=11, fontweight='bold')

# Panel 1: Train vs Valid Loss
ax = axes[0]
if tr_loss:
    ax.plot(epochs[:len(tr_loss)], tr_loss, 'b-o', ms=4, lw=1.8, label='Train Loss')
if val_loss:
    ax.plot(val_ep, val_loss, 'r-s', ms=5, lw=2.0, label='Valid Loss',
            markerfacecolor='white', markeredgewidth=2)
ax.set(xlabel='Epoch', ylabel='Loss', title='Train Loss vs Valid Loss')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_xlim(left=1)

# Panel 2: PSNR
ax = axes[1]
if psnr_vals:
    ax.plot(val_ep[:len(psnr_vals)], psnr_vals, 'g-D', ms=5, lw=1.8,
            label=f'PSNR (max={max(psnr_vals):.2f}dB)', color='seagreen')
ax.set(xlabel='Epoch', ylabel='PSNR (dB)', title='Validation PSNR')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_xlim(left=1)

# Panel 3: SSIM
ax = axes[2]
if ssim_vals:
    ax.plot(val_ep[:len(ssim_vals)], ssim_vals, '-^', ms=5, lw=1.8,
            label=f'SSIM (max={max(ssim_vals):.4f})', color='darkorange')
ax.set(xlabel='Epoch', ylabel='SSIM', title='Validation SSIM', ylim=(0,1))
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_xlim(left=1)

plt.tight_layout()
fig_path = os.path.join(exp_dir, 'loss_curve.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

if psnr_vals:
    print(f'Best PSNR: {max(psnr_vals):.4f} dB | Best SSIM: {max(ssim_vals):.6f}')

## 5. Export Excel + ZIP Hasil Training

In [ ]:
import zipfile

excel_path = os.path.join(exp_dir, 'training_results.xlsx')
history.export_excel(excel_path)

# ZIP hasil training — TANPA .pth
zip_path = os.path.join(OUTPUT_DIR, f'{exp_name}_training_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path(exp_dir).rglob('*'):
        if p.is_file() and p.suffix != '.pth' and 'checkpoint' not in p.name:
            if str(p) != zip_path:
                zf.write(p, p.relative_to(exp_dir))

model_mb = os.path.getsize(os.path.join(exp_dir, 'generator_final.pth')) / 1e6
zip_mb   = os.path.getsize(zip_path) / 1e6
print(f'[ZIP {zip_mb:.1f}MB] {Path(zip_path).name}  ← download ini (hasil training)')
print(f'[PTH {model_mb:.0f}MB] generator_final.pth   ← opsional (untuk inferensi lokal)')

## 6. Inferensi Test Set

In [ ]:
from inference import run_inference

summary, records = run_inference(
    model_path   = os.path.join(exp_dir, 'generator_final.pth'),
    test_hr_dir  = os.path.join(DATASET_ROOT, 'RGB'),
    output_dir   = os.path.join(OUTPUT_DIR, f'{exp_name}_inference'),
    experiment   = EXPERIMENT,
    num_rrdb     = NUM_RRDB,
    scale_factor = 4,
    tile_size    = 512,      # WAJIB untuk gambar 5280x3956
    file_list    = TEST_LIST,
    save_visuals = True,
)
print(f'Avg PSNR: {summary["avg_psnr_db"]:.4f} dB | Avg SSIM: {summary["avg_ssim"]:.6f}')

## 7. Contoh Visual SR + ZIP Inferensi

In [ ]:
infer_dir = os.path.join(OUTPUT_DIR, f'{exp_name}_inference')
vis_files = sorted(glob.glob(os.path.join(infer_dir, 'visualizations', '*.png')))
print(f'{len(vis_files)} visualisasi tersedia')

from PIL import Image as PILImage
for vf in vis_files[:3]:
    img = PILImage.open(vf)
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.imshow(img); ax.set_title(Path(vf).stem, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

# ZIP inferensi
infer_zip = os.path.join(OUTPUT_DIR, f'{exp_name}_inference_results.zip')
with zipfile.ZipFile(infer_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path(infer_dir).rglob('*'):
        if p.is_file() and p.suffix != '.pth':
            zf.write(p, p.relative_to(infer_dir))
print(f'[ZIP {os.path.getsize(infer_zip)/1e6:.1f}MB] {Path(infer_zip).name}')
print('Semua file siap di tab Output Kaggle!')

---
## 8. Evaluasi Segmentasi (Tabel 3.2)
**YOLOv8n-seg** — fine-tune 3 epoch, backbone frozen, satu model untuk semua skenario SEG1–SEG5

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
print('ultralytics: OK')


In [ ]:
# ── Step 1: Konversi mask WeedyRice → YOLO format ──────────
from segmentation.yolo_dataset_builder import build_yolo_dataset

YOLO_DS  = '/kaggle/working/yolo_dataset'
SEG_OUT  = '/kaggle/working/segmentation'

data_yaml = build_yolo_dataset(
    dataset_root    = DATASET_ROOT,
    output_dir      = YOLO_DS,
    train_list_path = TRAIN_LIST,
    val_list_path   = VAL_LIST,
    test_list_path  = TEST_LIST,
    copy_images     = False,   # symlink hemat disk
    min_area_px     = 200,
)
print(f'data.yaml: {data_yaml}')


In [ ]:
# ── Step 2: Fine-tune YOLOv8n-seg (3 epoch, backbone frozen) ──
# Model yang sama dipakai untuk SEMUA skenario SEG1-SEG5
# → perbandingan antar skenario adil

YOLO_FT_DIR  = '/kaggle/working/yolo_finetune'
YOLO_FT_CKPT = f'{YOLO_FT_DIR}/yolov8n_weed_best.pt'

if not os.path.isfile(YOLO_FT_CKPT):
    from segmentation.yolo_finetune import finetune_yolov8n_seg
    YOLO_FT_CKPT = finetune_yolov8n_seg(
        data_yaml  = data_yaml,
        output_dir = YOLO_FT_DIR,
        epochs     = 3,          # 1-3 epoch cukup (backbone frozen)
        img_size   = 640,
        batch_size = 8,
        device     = '',
    )
else:
    print(f'Model sudah ada: {YOLO_FT_CKPT}')

print(f'YOLOv8n-seg model: {YOLO_FT_CKPT}')


In [ ]:
# ── Step 3: Evaluasi SEG1–SEG5 ─────────────────────────────
from segmentation.seg_eval import run_all_scenarios

all_summaries = run_all_scenarios(
    dataset_root    = DATASET_ROOT,
    sr_results_base = OUTPUT_DIR,      # /kaggle/working/experiments
    yolo_model_path = YOLO_FT_CKPT,
    output_dir      = os.path.join(SEG_OUT, 'evaluation'),
    test_list       = TEST_LIST,
    device          = '',
    save_visuals    = True,
)


In [ ]:
# ── Tabel Ringkasan & Grafik ────────────────────────────────
import pandas as pd
from IPython.display import display

rows = [
    {'Skenario': k,
     'mIoU': v.get('mIoU', 0),
     'IoU Gulma': v.get('iou_weed', 0),
     'Precision': v.get('precision', 0),
     'Recall': v.get('recall', 0),
     'F1': v.get('F1', 0)}
    for k, v in all_summaries.items() if v
]
df = pd.DataFrame(rows)

print('\nHasil Evaluasi Segmentasi — Tabel 3.2')
fmt = {c: '{:.4f}' for c in df.columns if c != 'Skenario'}
display(
    df.style
    .highlight_max(subset=['mIoU','IoU Gulma','F1'], color='#c8e6c9')
    .highlight_min(subset=['mIoU','IoU Gulma','F1'], color='#ffcdd2')
    .format(fmt)
)

# Tampilkan grafik
fig_path = os.path.join(SEG_OUT, 'evaluation', 'segmentation_comparison.png')
if os.path.isfile(fig_path):
    from PIL import Image as PILImage
    plt.figure(figsize=(14, 5))
    plt.imshow(PILImage.open(fig_path))
    plt.axis('off'); plt.tight_layout(); plt.show()

# Tampilkan contoh overlay visual (3 gambar pertama dari SEG4)
import glob
overlays = sorted(glob.glob(
    os.path.join(SEG_OUT, 'evaluation', 'SEG4_UAV_IRE_Full',
                 'visualizations', '*.png')))[:3]
for ov in overlays:
    plt.figure(figsize=(15, 5))
    plt.imshow(PILImage.open(ov))
    plt.title(Path(ov).stem, fontsize=9)
    plt.axis('off'); plt.tight_layout(); plt.show()


In [ ]:
# ── ZIP hasil segmentasi ────────────────────────────────────
import zipfile

seg_zip = '/kaggle/working/segmentation_results.zip'
with zipfile.ZipFile(seg_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path(os.path.join(SEG_OUT, 'evaluation')).rglob('*'):
        if p.is_file() and p.suffix not in {'.pt', '.pth'}:
            if '_temp_LR' not in str(p) and str(p) != seg_zip:
                zf.write(p, p.relative_to(SEG_OUT))

sz = os.path.getsize(seg_zip) / 1e6
print(f'[ZIP {sz:.1f} MB] segmentation_results.zip')
print('  Isi: Excel mIoU SEG1-SEG5, grafik bar chart, overlay visual gulma')
print('  Download dari tab Output Kaggle!')
